# ⚙️ 03 — Feature Engineering
Pipeline Silver (nettoyage) → Gold (features ML-ready) + split train/test.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE HOUSE_PRICE_DB").collect()

# SILVER : encodage binaire yes/no -> 0/1
session.sql("""
CREATE OR REPLACE TABLE SILVER.HOUSE_PRICE_CLEAN AS
SELECT
    PRICE, AREA, BEDROOMS, BATHROOMS, STORIES,
    CASE WHEN UPPER(MAINROAD) = 'YES' THEN 1 ELSE 0 END AS MAINROAD,
    CASE WHEN UPPER(GUESTROOM) = 'YES' THEN 1 ELSE 0 END AS GUESTROOM,
    CASE WHEN UPPER(BASEMENT) = 'YES' THEN 1 ELSE 0 END AS BASEMENT,
    CASE WHEN UPPER(HOTWATERHEATING) = 'YES' THEN 1 ELSE 0 END AS HOTWATERHEATING,
    CASE WHEN UPPER(AIRCONDITIONING) = 'YES' THEN 1 ELSE 0 END AS AIRCONDITIONING,
    PARKING,
    CASE WHEN UPPER(PREFAREA) = 'YES' THEN 1 ELSE 0 END AS PREFAREA,
    LOWER(TRIM(FURNISHINGSTATUS)) AS FURNISHINGSTATUS,
    _LOAD_FILENAME, _LOAD_TIMESTAMP
FROM BRONZE.HOUSE_PRICE_STRUCTURED
WHERE PRICE IS NOT NULL AND AREA IS NOT NULL AND BEDROOMS IS NOT NULL
""").collect()
print("Silver :", session.table("SILVER.HOUSE_PRICE_CLEAN").count(), "lignes")

In [ ]:
# GOLD : features derivees
session.sql("""
CREATE OR REPLACE TABLE GOLD.HOUSE_FEATURES AS
SELECT
    PRICE, AREA, BEDROOMS, BATHROOMS, STORIES,
    MAINROAD, GUESTROOM, BASEMENT, HOTWATERHEATING,
    AIRCONDITIONING, PARKING, PREFAREA,
    CASE FURNISHINGSTATUS
        WHEN 'furnished' THEN 2
        WHEN 'semi-furnished' THEN 1
        ELSE 0
    END AS FURNISHING_ENCODED,
    AREA * BEDROOMS AS AREA_BEDS,
    BEDROOMS + BATHROOMS AS TOTAL_ROOMS,
    CASE WHEN AIRCONDITIONING = 1 AND PREFAREA = 1 THEN 1 ELSE 0 END AS PREMIUM_FLAG
FROM SILVER.HOUSE_PRICE_CLEAN
""").collect()
print("Gold :", session.table("GOLD.HOUSE_FEATURES").count(), "lignes")
session.table("GOLD.HOUSE_FEATURES").show(5)

In [ ]:
# Split train / test 80-20 (random sampling dans Snowflake)
import random; seed = 42
session.sql("""
CREATE OR REPLACE TABLE ML.TRAIN_SET AS
SELECT * FROM GOLD.HOUSE_FEATURES SAMPLE (80) SEED (42)
""").collect()
session.sql("""
CREATE OR REPLACE TABLE ML.TEST_SET AS
SELECT g.* FROM GOLD.HOUSE_FEATURES g
LEFT JOIN ML.TRAIN_SET t ON g.PRICE = t.PRICE AND g.AREA = t.AREA AND g.BEDROOMS = t.BEDROOMS
WHERE t.PRICE IS NULL
""").collect()
print("Train:", session.table("ML.TRAIN_SET").count(), " | Test:", session.table("ML.TEST_SET").count())